# Figure 7 — Detection performance against the Euclid Q1 MER catalogue

Catalogue-based complement to the injection-truth figure (Fig 6). The detection head is
the CenterNet on the **frozen** foundation's fused 10-band bottleneck (`vis_peak`, round-1,
200 tiles). Evaluated on the 16 held-out tiles, matched to the clean MER catalogue
(`vis_det & !spurious`) within $0.5''$:

- **(a) Completeness vs MER VIS magnitude** — how much of the existing Euclid catalogue we
  recover; 50% and 90% completeness marked.
- **(b) Completeness vs measured VIS S/N** — the same recovery binned by the *measured*
  aperture S/N of each source, so it does not depend on a magnitude zeropoint.
- **(c) Operating point** — completeness (to VIS $<24.5$) and purity as the detection
  threshold is swept; our $0.30$ working point is marked. Purity is measured against the
  *full* MER catalogue and is therefore a conservative floor — some unmatched detections are
  real sources MER's VIS-driven catalogue missed (see injection figure / text).

`REGEN` runs the frozen detector over the eval tiles (GPU, single pass over confs) and
caches to `_fig7_completeness_cache.pkl`; the plot cell loads it. Saves
`paper/figures/fig7_completeness.png`.

In [ ]:
import sys, pickle
from pathlib import Path
import numpy as np

def find_repo_root(s=None):
    s=(s or Path.cwd()).resolve()
    for c in [s,*s.parents]:
        if (c/'models').exists() and (c/'data').exists(): return c
    raise FileNotFoundError('repo root')
REPO=find_repo_root(); NB=Path.cwd()
for p in [REPO/'models',REPO/'models'/'detection',REPO/'models'/'astrometry2']:
    if str(p) not in sys.path: sys.path.insert(0,str(p))
CACHE=NB/'_fig7_completeness_cache.pkl'
OUT=REPO/'paper'/'figures'/'fig7_completeness.png'
MER_FITS=REPO/'data'/'edf_s_ood'/'catalogs_compact'/'mer_FINAL_q1_ECDFS_footprint.fits'
DET_CKPT=REPO/'checkpoints'/'centernet_v10_vispeak_r1_full'/'centernet_best.pt'
ENC=REPO/'models'/'checkpoints'/'jaisp_v10_warmstart'/'checkpoint_best.pt'
EUCLID_DIR=REPO/'data'/'euclid_tiles_200'; RUBIN_DIR=REPO/'data'/'rubin_tiles_200'
EVAL_STEMS=(REPO/'data'/'bakeoff_eval16_stems.txt').read_text().split()

CONFS=(0.15,0.20,0.25,0.30,0.35,0.40,0.50)   # threshold sweep for the operating-point panel
CONF_MAIN=0.30                                 # working point used in panels (a),(b) and the paper
RAD_AS=0.5; MARGIN=8; COMP_MAGLIM=24.5         # bright limit for completeness in panel (c)
REGEN=not CACHE.exists()
print(f'detector: {DET_CKPT.parent.name}\nCACHE exists={CACHE.exists()}  REGEN={REGEN}')

In [ ]:
if REGEN:
    import glob, torch
    from scipy.spatial import cKDTree
    from detection.validation_utils import (load_detector, load_mer, build_inputs,
                                            run_detect, _wcs_vis, per_band_snr, PXE)
    DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    mer=load_mer(str(MER_FITS))
    det,_=load_detector(str(DET_CKPT), str(ENC), device=DEV)
    rpx=RAD_AS/PXE

    # per-conf accumulators for the operating-point panel
    comp_hit={c:0 for c in CONFS}; comp_tot={c:0 for c in CONFS}     # completeness, MER mag<COMP_MAGLIM
    pur_hit ={c:0 for c in CONFS}; pur_tot ={c:0 for c in CONFS}     # purity, detections vs full MER
    # per-source records at CONF_MAIN for panels (a) completeness-vs-mag and (b) completeness-vs-SNR
    rec_mag=[]; rec_hit=[]; rec_snr=[]

    for stem in EVAL_STEMS:
        ep=f'{EUCLID_DIR}/{stem}_euclid.npz'; rp=f'{RUBIN_DIR}/{stem}.npz'
        if not (glob.glob(ep) and glob.glob(rp)): continue
        ed=dict(np.load(ep,allow_pickle=True)); rd=dict(np.load(rp,allow_pickle=True))
        images,rms,vh=build_inputs(ed,rd); H,W=vh; vw=_wcs_vis(ed)
        # clean MER inside tile (completeness reference) + VIS mag
        cx,cy=vw.all_world2pix(mer['cRA'],mer['cDEC'],0)
        ck=(cx>=MARGIN)&(cx<W-MARGIN)&(cy>=MARGIN)&(cy<H-MARGIN)&np.isfinite(mer['cMAG'])
        cmer=np.c_[cx[ck],cy[ck]]; cmag=mer['cMAG'][ck]
        # full MER inside tile (purity reference)
        fx,fy=vw.all_world2pix(mer['fRA'],mer['fDEC'],0)
        fk=(fx>=MARGIN)&(fx<W-MARGIN)&(fy>=MARGIN)&(fy<H-MARGIN)
        fmer=np.c_[fx[fk],fy[fk]]
        if len(cmer)<2 or len(fmer)<2: continue
        # measured VIS aperture S/N at each clean-MER position (for panel b)
        snr_vis=per_band_snr(ed,rd,vw,cmer)['euclid_VIS']
        ftree=cKDTree(fmer)
        for conf in CONFS:
            D=run_detect(det,images,rms,vh,DEV,conf)
            din=(D[:,0]>=MARGIN)&(D[:,0]<W-MARGIN)&(D[:,1]>=MARGIN)&(D[:,1]<H-MARGIN)
            D=D[din]
            if len(D)<2:
                comp_tot[conf]+=int((cmag<COMP_MAGLIM).sum()); continue
            dtree=cKDTree(D)
            d_m,_=dtree.query(cmer); hit=d_m<rpx
            # completeness (mag<lim) and purity for this conf
            bright=cmag<COMP_MAGLIM
            comp_hit[conf]+=int(hit[bright].sum()); comp_tot[conf]+=int(bright.sum())
            d_o,_=ftree.query(D); pur_hit[conf]+=int((d_o<rpx).sum()); pur_tot[conf]+=len(D)
            if abs(conf-CONF_MAIN)<1e-6:
                rec_mag.append(cmag); rec_hit.append(hit); rec_snr.append(snr_vis)

    rec_mag=np.concatenate(rec_mag); rec_hit=np.concatenate(rec_hit); rec_snr=np.concatenate(rec_snr)
    comp_curve={c:(comp_hit[c]/max(comp_tot[c],1)) for c in CONFS}
    pur_curve ={c:(pur_hit[c]/max(pur_tot[c],1))  for c in CONFS}
    pickle.dump(dict(rec_mag=rec_mag,rec_hit=rec_hit,rec_snr=rec_snr,
                     confs=list(CONFS),comp_curve=comp_curve,pur_curve=pur_curve,
                     conf_main=CONF_MAIN,comp_maglim=COMP_MAGLIM,rad_as=RAD_AS,
                     n_tiles=len(EVAL_STEMS),det=DET_CKPT.parent.name),open(CACHE,'wb'))
    print(f'cached -> {CACHE}  n_clean_MER={len(rec_mag)}  purity@{CONF_MAIN}={pur_curve[CONF_MAIN]:.3f}')
    del det
    if DEV.type=='cuda': torch.cuda.empty_cache()
else:
    print('REGEN skipped (cache present).')

In [ ]:
import pickle, numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

c=pickle.load(open(CACHE,'rb'))
mag,hit,snr=c['rec_mag'],c['rec_hit'].astype(float),c['rec_snr']
confs=np.array(c['confs']); comp=np.array([100*c['comp_curve'][k] for k in c['confs']])
pur =np.array([100*c['pur_curve'][k] for k in c['confs']]); cmain=c['conf_main']

def binned(x,y,edges,nmin=20):
    cen,val,n=[],[],[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        m=(x>=lo)&(x<hi)&np.isfinite(x)
        if m.sum()<nmin: continue
        cen.append(np.sqrt(lo*hi) if lo>0 and hi>0 else 0.5*(lo+hi))
        val.append(100*y[m].mean()); n.append(int(m.sum()))
    return np.array(cen),np.array(val),np.array(n)

def cross50(xc,yc):
    for i in range(len(xc)-1):
        if yc[i]>=50>=yc[i+1]:
            t=(50-yc[i])/((yc[i+1]-yc[i]) or -1e-9); return xc[i]+t*(xc[i+1]-xc[i])
    return None
def cross(level,xc,yc):
    for i in range(len(xc)-1):
        if yc[i]>=level>=yc[i+1]:
            t=(level-yc[i])/((yc[i+1]-yc[i]) or -1e-9); return xc[i]+t*(xc[i+1]-xc[i])
    return None

CA,CV,CP='#1f77b4','#2ca02c','#d62728'
fig=plt.figure(figsize=(14,4.6))
gs=GridSpec(1,3,wspace=0.27,left=0.055,right=0.985,top=0.88,bottom=0.16)

# (a) completeness vs MER VIS mag
ax=fig.add_subplot(gs[0,0])
medg=np.arange(19.5,27.01,0.5); mc,mv,mn=binned(mag,hit,medg,nmin=20)
ax.plot(mc,mv,'-o',color=CA,lw=2,ms=4)
ax.axhline(50,color='gray',ls=':',lw=1); ax.axhline(90,color='gray',ls=':',lw=1)
d50=cross(50,mc,mv); d90=cross(90,mc,mv)
if d90: ax.scatter([d90],[90],s=70,facecolors='white',edgecolors=CA,zorder=5,lw=1.6); ax.text(d90,93,f'90%: {d90:.1f}',color=CA,fontsize=9,ha='center')
if d50: ax.scatter([d50],[50],s=70,facecolors='white',edgecolors=CA,zorder=5,lw=1.6); ax.text(d50,53,f'50%: {d50:.1f}',color=CA,fontsize=9,ha='center')
ax.set_xlabel('MER VIS magnitude'); ax.set_ylabel('completeness [%]'); ax.set_ylim(0,103)
ax.set_title('(a)  Completeness vs MER catalogue',loc='left',fontsize=11); ax.grid(alpha=0.25)

# (b) completeness vs measured VIS S/N
ax=fig.add_subplot(gs[0,1])
sedg=np.logspace(np.log10(1.5),np.log10(120),13); sc,sv,sn=binned(snr,hit,sedg,nmin=20)
ax.plot(sc,sv,'-s',color=CV,lw=2,ms=4)
ax.axhline(50,color='gray',ls=':',lw=1)
s50=cross50(sc,sv)
if s50: ax.axvline(s50,color=CV,ls='--',lw=1.2); ax.text(s50*1.05,8,f'50% at S/N={s50:.1f}',color=CV,fontsize=9)
ax.set_xscale('log'); ax.set_xlabel('measured VIS aperture S/N'); ax.set_ylabel('completeness [%]'); ax.set_ylim(0,103)
ax.set_title('(b)  Completeness vs measured S/N',loc='left',fontsize=11); ax.grid(alpha=0.25,which='both')

# (c) operating point: completeness & purity vs threshold
ax=fig.add_subplot(gs[0,2])
ax.plot(confs,comp,'-o',color=CA,lw=2,ms=4,label=f'completeness (VIS<{c["comp_maglim"]:.1f})')
ax.plot(confs,pur ,'-^',color=CP,lw=2,ms=4,label='purity (vs MER, floor)')
ax.axvline(cmain,color='k',ls='--',lw=1.1)
ax.text(cmain,2,f'  working point {cmain:.2f}',fontsize=8.5,rotation=90,va='bottom',ha='left')
ax.set_xlabel('detection threshold'); ax.set_ylabel('percent [%]'); ax.set_ylim(0,103)
ax.set_title('(c)  Operating point',loc='left',fontsize=11)
ax.legend(fontsize=8.5,loc='lower center'); ax.grid(alpha=0.25)

fig.savefig(OUT,dpi=200,bbox_inches='tight',facecolor='white')
print('saved ->',OUT,'| 50%%/90%% mag=%.2f/%.2f | S/N50=%.1f | purity@%.2f=%.1f%%'%(
      d50 or -1,d90 or -1,s50 or -1,cmain,100*c['pur_curve'][cmain]))
plt.show()